In [2]:
import pandas as pd
import numpy as np

#resturants
df_rests = pd.read_csv(
    "../Project/team_project_dataset/restaurants_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_rests.columns = df_rests.iloc[0]
df_rests = df_rests[1:]
df_rests.reset_index(drop=True)

#test
df_test = pd.read_csv(
    "../Project/team_project_dataset/test_reviews_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_test.columns = df_test.iloc[0]
df_test = df_test[1:]
df_test.reset_index(drop=True)

#train
df_train = pd.read_csv(
    "../Project/team_project_dataset/train_reviews_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_train.columns = df_train.iloc[0]
df_train = df_train[1:]
df_train.reset_index(drop=True)


,review_id,user_id,business_id,stars,datetime,text
0,ixa--i8zAivs8rD10tleZQ,-0EcgtUXe1rzrkmdih_tYg,VgAKmXE8B7J0I_O_R13UKQ,4.0,2018-07-03 03:24:32,I lived here before and not once that I came t...
1,MHj1HEPM5Bv7_UhoiwOSGA,-0EcgtUXe1rzrkmdih_tYg,YrNtBUOUOYwmRZ_UVwH8iQ,4.0,2018-07-04 15:47:17,Totally a great experienced. Been coming to S...
2,_sTp9AsEu60ORSqqb9juJg,-0EcgtUXe1rzrkmdih_tYg,Aes-0Q_guDeYewMapFs_vg,4.0,2018-07-06 05:38:42,"Great outdoor seating, the servers are attenti..."
3,2Tcu86xzIADtfuy5jeio9A,-0EcgtUXe1rzrkmdih_tYg,9ugpNKKhnYRa51qXoxUw_A,3.0,2018-07-06 05:49:49,Went for the first time. The lady in the fron...
4,J6X5I_LQnii7QeIW79gBVg,-0EcgtUXe1rzrkmdih_tYg,DOfiulOub9hVPBCtiDl9Fw,3.0,2018-07-06 06:05:15,"Pizza was great, servers are friendly and atte..."
...,...,...,...,...,...,...
41576,hJ6_1V1gLMuWMUfPeaGomA,zyt0joW7uNeQof5tthQAHg,JjmmSW_QQh2Db4fuIEMATA,4.0,2013-09-08 02:32:56,I never had such a good pizza so fast! We had ...
41577,wqUCvQf_bP_gQxrFRz94LA,zyt0joW7uNeQof5tthQAHg,Aes-0Q_guDeYewMapFs_vg,3.0,2013-09-09 03:56:41,We had lunch at the longboard grill. The food ...
41578,CufxkovWUbBqcj8avCai_g,zyt0joW7uNeQof5tthQAHg,0CHIbqSkGWBr2KMkIUocEA,4.0,2013-09-09 04:31:33,So first let me say that this restaurant is in...
41579,0yQfK9Ci7fbX850-LEyJbw,zyt0joW7uNeQof5tthQAHg,U2hkeMI-q4cS35QolJYN0A,2.0,2013-09-10 02:46:32,"So if this is, as other reviewers state, the b..."


In [3]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
# Multi-hot encode restaurant categories using get_dummies

# Split comma-separated categories string into a list per restaurant
df_rests["category_list"] = df_rests["categories"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",")]
)

# get_dummies for binary encoding
category_dummies = pd.get_dummies(
    df_rests["category_list"].explode()
).groupby(level=0).max()
category_dummies = category_dummies.reindex(df_rests.index, fill_value=0)

In [4]:
#on top of the categorical embeddings, I want to add to each embedding a few values connoting attribute values

#first clean values for the attribute column
import ast
from sklearn.preprocessing import normalize
from scipy.sparse import hstack, csr_matrix

def str_to_dict(s):
    """
    Safely parse a string representation of a Python dictionary.
    Args:
        s (str): A string that may contain a dictionary literal.
    Returns:
        dict: The parsed dictionary, or an empty dict if parsing fails.
    """
    try:
        return ast.literal_eval(s)
    except: #need this or else it won't run
        return {}
df_rests['attributes_dict'] = df_rests['attributes'].apply(str_to_dict)

def clean_values(d):
    """
    Strip extra quote characters from string values in a dictionary.
    Args:
        d (dict): A dictionary whose values may contain leading/trailing quotes.
    Returns:
        dict: The same dictionary with string values without quote characters.
    """
    return {k: (v.strip("'\"") if isinstance(v, str) else v) for k, v in d.items()}
df_rests['attributes_dict'] = df_rests['attributes_dict'].apply(clean_values)

# use RestaurantsPriceRange2, GoodForKids, and RestaurantsAttire; these are all good indicators because people feel strongly about prices, about their families, and about "comfortability" of a restaurant. A person who wants to eat cheap will really appreciate a cheap recommendation, a person who has kids will really appreciate a restaurant that caters to them, and a person who likes to be comfortable/homey will really appreciate a restaurant recommendation that feels like home.

# Binary attribute: GoodForKids
df_rests["good_for_kids"] = df_rests["attributes_dict"].apply(
    lambda d: 1 if d.get("GoodForKids") == "True" else 0
)

# Binary attribute: RestaurantsAttire (casual vs not)
df_rests["casual"] = df_rests["attributes_dict"].apply(
    lambda d: 1 if d.get("RestaurantsAttire") == "casual" else 0
)

# Multi-class attribute: PriceRange — one-hot encode with get_dummies
# Restaurants with no price data are assigned "unknown" category
df_rests["price_range"] = df_rests["attributes_dict"].apply(
    lambda d: d.get("RestaurantsPriceRange2", "unknown")
)
price_dummies = pd.get_dummies(df_rests["price_range"], prefix="price")
price_dummies = price_dummies.reindex(df_rests.index, fill_value=0)


In [5]:
X_cat = csr_matrix(category_dummies.values.astype(float))
X_attr = csr_matrix(price_dummies.values.astype(float))
X_binary = csr_matrix(df_rests[["good_for_kids", "casual"]].values.astype(float))

# set weights
weight_cat = 0.6
weight_attr = 0.25
weight_binary = 0.15

# Concatenate weighted blocks into a single matrix
X_meta_sparse = hstack([X_cat * weight_cat, X_attr * weight_attr, X_binary * weight_binary])
# L2 normalize each row so all metadata vectors have unit length
X_meta = normalize(X_meta_sparse, norm="l2", axis=1).toarray()

df_rests["meta_embedding"] = list(X_meta)
print("Metadata vector size:", X_meta.shape[1])

Metadata vector size: 232


In [6]:
#NOW WE NEED USER PROFILES FOR EACH USER IN THE df_train

#we want to include ranking, although we are not including review text
#to implement this, we want to get a rating bias for each restaurant (whether it was rated more positively or negatively than the user average) and then use that to weight the restaurant vectors in the user profile (so that the recommendations we get are more similar to the restaurants the user liked more)
#first get the mean rating for each user
df_train['stars'] = pd.to_numeric(df_train['stars'])
user_means = df_train.groupby("user_id")["stars"].mean()
df_train = df_train.merge(user_means.rename("user_mean"), on="user_id")

In [7]:
#now get the rating bias for each user
df_train["adjusted_rating"] = df_train["stars"] - df_train["user_mean"]

In [8]:
#put everything together
merge = df_train.merge(df_rests, on="business_id")

In [9]:
#now we want to get the user profiles
merge = merge.set_index("business_id")
user_profiles = {}
for user, group in df_train.groupby("user_id"):
    vectors = merge.loc[group["business_id"], "meta_embedding"]
    user_profiles[user] = np.vstack(vectors).mean(axis=0)


In [12]:
#get recommendations
from sklearn.metrics.pairwise import cosine_similarity

def recommend_restaurants(user_id, user_embeddings, restaurants_df, k):
    user_vec = user_embeddings[user_id].reshape(1, -1)
    restaurant_matrix = np.vstack(restaurants_df['meta_embedding'].values)

    similarities = cosine_similarity(user_vec, restaurant_matrix)[0]
    top_k_idx = np.argsort(similarities)[-k:][::-1]

    results = restaurants_df[['business_id', 'name']].iloc[top_k_idx].copy()
    results['similarity'] = similarities[top_k_idx]

    return results

In [13]:
#test
ay = recommend_restaurants('-0EcgtUXe1rzrkmdih_tYg', user_profiles, df_rests, 10)

In [15]:
#now we can do accuracy metrics. first is NDCG@k
def ndcg_at_k(recommended_ids, true_id, k):
    try:
        rank = recommended_ids.index(true_id) + 1
    except ValueError:
        return 0.0

    return 1 / np.log2(rank + 1)

def evaluate_ndcg(user_embeddings, restaurants_df, test_df, k):
    scores = []

    for _, row in test_df.iterrows():
        user_id = row['user_id']
        true_business_id = row['business_id']

        recs = recommend_restaurants(user_id,user_embeddings,restaurants_df,k)
        recommended_ids = recs['business_id'].tolist()
        score = ndcg_at_k(recommended_ids, true_business_id, k)
        scores.append(score)

    return np.mean(scores)

#then hit@k
def hit_at_k(recommended_ids, true_id):
    return int(true_id in recommended_ids)

import numpy as np

def evaluate_hit_at_k(user_embeddings, restaurants_df, test_df, k):
    hits = []
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        true_business_id = row['business_id']
        recs = recommend_restaurants(user_id, user_embeddings, restaurants_df, k)

        recommended_ids = recs['business_id'].tolist()

        hit = hit_at_k(recommended_ids, true_business_id)
        hits.append(hit)

    return np.mean(hits)

In [16]:
for k in [10, 20, 30]:
    print(f"Hit@{k}:", evaluate_hit_at_k(user_profiles, df_rests, df_test, k), f"\t\tNDCG@{k}:", evaluate_ndcg(user_profiles, df_rests, df_test, k))

Hit@10: 0.017912934805248908 		NDCG@10: 0.007291341608453759
Hit@20: 0.039158508644032496 		NDCG@20: 0.012667980724110094
Hit@30: 0.06019579254322016 		NDCG@30: 0.017132865083585514
